In [ ]:
# ============================================================
# Block A (NOCOUPL O3): 最终修复版 (cftime 硬核偏移 + 物理 1-103 接 002 偏移)
# ============================================================
import os
import glob
import numpy as np
import xarray as xr

# --- 配置区域 ---
DATA_DIR_NOCOUPL = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/O3"
OUT_ROOT = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

PRESSURE_RANGES = [(30, 70, "30_70hPa"), (1, 100, "1_100hPa")]

def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa):
    """混合层 O3 柱含量积分逻辑"""
    g, M_air, Na = 9.80665, 28.964 / 1000.0, 6.02214e23
    factor = Na / (g * M_air * 2.687e20)

    P0, PS = ds_sub["P0"], ds_sub["PS"]
    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = (
        P_interface.isel(ilev=slice(0, -1))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )
    p_ip1 = (
        P_interface.isel(ilev=slice(1, None))
        .rename({"ilev": "lev"})
        .assign_coords(lev=ds_sub["lev"])
    )

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    pT, pB = p_top_hpa * 100.0, p_bot_hpa * 100.0
    upper = xr.where(p_layer_top > pT, p_layer_top, pT)
    lower = xr.where(p_layer_bot < pB, p_layer_bot, pB)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    return (ds_sub["O3"] * overlap * factor).sum(dim="lev")

def process_b2000wcn_nocoupl():
    exp_out_dir = os.path.join(OUT_ROOT, "B2000WCN_NOCOUPL")
    os.makedirs(exp_out_dir, exist_ok=True)

    all_files = sorted(glob.glob(os.path.join(DATA_DIR_NOCOUPL, "B2000WCN.NOCOUPL.sample.cam.h3.*.O3.nc")))

    # 物理筛选：
    # run001 -> 1-103
    # run002 -> 105-205
    # 104 明确跳过（过渡/残缺年）
    files_001 = [f for f in all_files if 1 <= int(os.path.basename(f).split(".")[-3]) <= 103]
    files_002 = [f for f in all_files if 105 <= int(os.path.basename(f).split(".")[-3]) <= 205]

    print(f"[B2000WCN_NOCOUPL] Loading Run 001 (Y1-103)...")
    ds1 = xr.open_mfdataset(files_001, combine="by_coords")

    print(f"[B2000WCN_NOCOUPL] Loading Run 002 (Y105-205) and shifting internally to Y104-Y204...")
    ds2 = xr.open_mfdataset(files_002, combine="by_coords")

    # --- 修正 cftime 时间轴：内部 Y1 -> 物理 Y104 ---
    offset = 103
    old_times = ds2.time.values
    new_times = [t.replace(year=t.year + offset) for t in old_times]
    ds2 = ds2.assign_coords(time=new_times)

    # 合并：此时 ds1 到 103 年，ds2 从 104 年开始（时间上）
    ds = xr.concat([ds1, ds2], dim="time").sortby("time")

    # 60-90N 极区平均
    ds_polar = ds.sel(lat=slice(60, 90))
    weights = np.cos(np.deg2rad(ds_polar.lat))

    for ptop, pbot, tag in PRESSURE_RANGES:
        print(f"  Calculating {tag}...")
        o3_ts = (
            calc_parc_o3_hybrid(ds_polar, ptop, pbot)
            .weighted(weights)
            .mean(dim=["lat", "lon"])
            .compute()
        )

        out_nc = os.path.join(exp_out_dir, f"O3_pc_B2000WCN_NOCOUPL_{tag}.nc")
        o3_ts.to_netcdf(out_nc)

        # --- 5-day running mean (Friedel-style) ---
        o3_5d = o3_ts.rolling(time=5, center=True, min_periods=5).mean()

        # 只在 3-4 月找 minimum（用 5d 平均后的日序列）
        spring_all = o3_5d.sel(time=o3_5d.time.dt.month.isin([3, 4]))

        # 有效年份：3-4 月至少 ~58 个有效 5-day window 日
        day_counts = spring_all.groupby("time.year").count()
        valid_yrs = day_counts.year.where(day_counts >= 58, drop=True)

        spring_min = spring_all.groupby("time.year").min().sel(year=valid_yrs)

        # 保存 extreme-year txt
        idx_sort = np.argsort(spring_min.values)

        np.savetxt(
            os.path.join(exp_out_dir, f"lowest10_years_sorted_{tag}.txt"),
            spring_min.year.values[idx_sort[:10]],
            fmt="%04d"
        )

        np.savetxt(
            os.path.join(exp_out_dir, f"low25pct_years_{tag}.txt"),
            spring_min.year.values[idx_sort[:max(1, int(0.25 * len(spring_min)))]],
            fmt="%04d"
        )

if __name__ == "__main__":
    process_b2000wcn_nocoupl()
    print("\n[Block A - NOCOUPL O3] All Done. Run 002 (internally Y1) is now physical Y104.")

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

import os
import glob
import numpy as np
import xarray as xr

# ============================================================
# Block A (NOCOUPL T): T_min_60_90N calculation
# ------------------------------------------------------------
# Diagnostic:
#   T(time, lev, lat, lon)
#   -> zonal mean over lon
#   -> select 60–90N
#   -> minimum over lat
#   -> T_min_60_90N(time, lev)
#
# Year handling:
#   Run 001: physical years 1–103
#   Run 002: file years 105–205, internal cftime years start from 1
#            shift internal time by +103 years
#            so Run 002 becomes physical years 104–204
#
# Output:
#   /home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data/B2000WCN_NOCOUPL/T_min_60_90N_B2000WCN_NOCOUPL.nc
# ============================================================

# -----------------------------
# Paths
# -----------------------------
RAW_DATA_ROOT = "/mnt/soclim0/public_data/weiji"
SAVE_BASE_DIR = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"

EXP_NAME = "B2000WCN_NOCOUPL"

T_DIR = os.path.join(RAW_DATA_ROOT, "B2000WCN_NOCOUPL001002", "T")

# 按照 NOCOUPL O3 的命名习惯推断：
# B2000WCN.NOCOUPL.sample.cam.h3.*.T.nc
PATTERN = "B2000WCN.NOCOUPL.sample.cam.h3.*.T.nc"

OUT_DIR = os.path.join(SAVE_BASE_DIR, "B2000WCN_NOCOUPL")
os.makedirs(OUT_DIR, exist_ok=True)

OUT_FILE = os.path.join(OUT_DIR, "T_min_60_90N_B2000WCN_NOCOUPL.nc")


# ============================================================
# Helper functions
# ============================================================

def get_year_from_filename(path):
    """
    Extract year token from file name.

    Expected example:
      B2000WCN.NOCOUPL.sample.cam.h3.0105.T.nc

    The year token is expected to be the third item from the end:
      split(".")[-3]
    """
    return int(os.path.basename(path).split(".")[-3])


def calc_minT_pcap(ds, var="T"):
    """
    Calculate polar-cap minimum temperature.

    Steps:
      1. zonal mean over longitude
      2. select 60–90N
      3. minimum over latitude
      4. rename vertical coordinate to lev
      5. convert lev to Pa if needed

    Output dimensions:
      time x lev
    """
    if var not in ds.data_vars:
        raise ValueError(f"Variable {var} not found. Available variables: {list(ds.data_vars)}")

    da = ds[var]

    # 1. Zonal mean
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    # 2. Select 60–90N
    # WACCM lat is normally ascending, but this is made slightly safer.
    if da["lat"].values[0] > da["lat"].values[-1]:
        da_cap = da.sel(lat=slice(90, 60))
    else:
        da_cap = da.sel(lat=slice(60, 90))

    # 3. Minimum over latitude
    da_min = da_cap.min(dim="lat")  # expected dims: time x lev

    # 4. Detect vertical coordinate
    lev_name = None
    for name in ("lev", "plev", "lev_p", "level"):
        if name in da_min.dims or name in da_min.coords:
            lev_name = name
            break

    if lev_name is None:
        raise ValueError(f"No vertical coordinate found in dims={da_min.dims}, coords={list(da_min.coords)}")

    # 5. Convert vertical coordinate to Pa
    lev_vals = da_min[lev_name].values.astype(float)
    max_val = float(np.nanmax(lev_vals))

    if max_val <= 2000.0:
        lev_pa = lev_vals * 100.0
        lev_unit_guess = "hPa"
    else:
        lev_pa = lev_vals
        lev_unit_guess = "Pa"

    if lev_name != "lev":
        da_min = da_min.rename({lev_name: "lev"})

    da_min = da_min.assign_coords(lev=lev_pa)

    da_min.name = "T_min_60_90N"
    da_min.attrs["units"] = "K"
    da_min.attrs["description"] = (
        "Polar-cap minimum temperature. "
        "Computed from zonal-mean T, then minimum over 60–90N "
        "at each time and vertical level."
    )
    da_min.attrs["lat_band"] = "60-90N"
    da_min.attrs["method"] = "zonal mean over longitude, then minimum over latitude"
    da_min.attrs["original_vertical_unit_guess"] = lev_unit_guess
    da_min["lev"].attrs["units"] = "Pa"

    return da_min


def process_nocoupl_t():
    print(f"[{EXP_NAME} T] Searching files in:")
    print(f"  {T_DIR}")
    print(f"[{EXP_NAME} T] Pattern:")
    print(f"  {PATTERN}")

    all_files = sorted(glob.glob(os.path.join(T_DIR, PATTERN)))

    if not all_files:
        raise FileNotFoundError(
            f"No T files found with pattern:\n{os.path.join(T_DIR, PATTERN)}"
        )

    print(f"[{EXP_NAME} T] Found {len(all_files)} files.")
    print(f"  First: {all_files[0]}")
    print(f"  Last : {all_files[-1]}")

    # ------------------------------------------------------------
    # Physical file selection
    # Consistent with NOCOUPL O3:
    #   Run 001 -> file years 1–103
    #   Run 002 -> file years 105–205
    #   Skip file year 104
    # ------------------------------------------------------------
    files_001 = [
        f for f in all_files
        if 1 <= get_year_from_filename(f) <= 103
    ]

    files_002 = [
        f for f in all_files
        if 105 <= get_year_from_filename(f) <= 205
    ]

    print(f"[{EXP_NAME} T] Run 001 files Y1–103:  {len(files_001)}")
    print(f"[{EXP_NAME} T] Run 002 files Y105–205: {len(files_002)}")

    if len(files_001) == 0:
        raise RuntimeError("No Run 001 files found.")
    if len(files_002) == 0:
        raise RuntimeError("No Run 002 files found.")

    # ------------------------------------------------------------
    # Load Run 001
    # ------------------------------------------------------------
    print(f"[{EXP_NAME} T] Loading Run 001 as physical Y1–103...")
    ds1 = xr.open_mfdataset(
        files_001,
        combine="by_coords",
        chunks={"time": 365, "lat": 48}
    )

    # ------------------------------------------------------------
    # Load Run 002 and shift internal time
    # ------------------------------------------------------------
    print(f"[{EXP_NAME} T] Loading Run 002 and shifting internally by +103 years...")
    ds2 = xr.open_mfdataset(
        files_002,
        combine="by_coords",
        chunks={"time": 365, "lat": 48}
    )

    # Run 002 内部时间从 year 1 开始；
    # 物理上接在 Run 001 的 103 年之后，因此加 103。
    offset = 103
    new_times = [t.replace(year=t.year + offset) for t in ds2.time.values]
    ds2 = ds2.assign_coords(time=new_times)

    # ------------------------------------------------------------
    # Merge
    # ------------------------------------------------------------
    print(f"[{EXP_NAME} T] Concatenating Run 001 + shifted Run 002...")
    ds = xr.concat([ds1, ds2], dim="time").sortby("time")

    print(f"[{EXP_NAME} T] Time range after merge:")
    print(f"  start = {ds.time.values[0]}")
    print(f"  end   = {ds.time.values[-1]}")
    print(f"  ntime = {ds.sizes['time']}")

    # ------------------------------------------------------------
    # Calculate T_min
    # ------------------------------------------------------------
    print(f"[{EXP_NAME} T] Calculating T_min_60_90N...")
    t_min_da = calc_minT_pcap(ds, var="T").compute()

    # ------------------------------------------------------------
    # Save
    # ------------------------------------------------------------
    print(f"[{EXP_NAME} T] Output DataArray:")
    print(t_min_da)

    print(f"[{EXP_NAME} T] Saving to:")
    print(f"  {OUT_FILE}")

    encoding = {
        "T_min_60_90N": {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",
        }
    }

    t_min_da = t_min_da.astype("float32")
    t_min_da.to_netcdf(OUT_FILE, encoding=encoding)

    ds1.close()
    ds2.close()
    ds.close()

    print(f"✅ Success: {EXP_NAME} T_min saved.")
    print(f"   {OUT_FILE}")


if __name__ == "__main__":
    process_nocoupl_t()
    print(f"\n[Block A - NOCOUPL T] All completed.")

In [ ]:
# ============================================================
# Block C (NOCOUPL Scatter Pro): 可定制时间窗与统计维度的 EP-O3 分析
# 修正版：跨年 Fz 季节（如 DJF）剔除连接年 104；JF vs MA 不受影响
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_NOCOUPL_EP = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

# 连接年（run001 和 run002 的拼接年）
BRIDGE_YEAR = 104

# 月份映射表（保留你的写法）
MONTH_MAP = {'J':1, 'F':2, 'M':3, 'A':4, 'm':5, 'j':6, 'x':7, 'a':8, 'S':9, 'O':10, 'N':11, 'D':12}

SHORT_MAP = {
    "DJF": [12, 1, 2],
    "JF":  [1, 2],
    "FM":  [2, 3],
    "MA":  [3, 4],
    "NDJ": [11, 12, 1],
    "JFM": [1, 2, 3],
    "OND": [10, 11, 12]
}

def is_cross_year_season(season_str):
    """判断一个 season 是否跨年（如 DJF / NDJ）"""
    months = SHORT_MAP.get(season_str, [12, 1, 2])
    return any(m in [11, 12] for m in months) and any(m in [1, 2] for m in months)

def get_seasonal_data(da, season_str, method="mean"):
    """
    处理跨年拼接逻辑并返回按年统计的时间序列
    season_str: 如 "DJF", "NDJ", "MA"
    method: 'mean' 或 'min'
    """
    months = SHORT_MAP.get(season_str, [12, 1, 2])  # 默认 DJF
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    cross_year = is_cross_year_season(season_str)

    for yr in all_years:
        try:
            if cross_year:
                # Year-1 的 late months + Year 的 early months
                early_months = [m for m in months if m < 6]
                late_months = [m for m in months if m >= 6]

                parts = []
                for m in late_months:
                    parts.append(ts[f"{int(yr-1):04d}-{m:02d}"])
                for m in early_months:
                    parts.append(ts[f"{int(yr):04d}-{m:02d}"])

                combined = pd.concat(parts)
            else:
                combined = pd.concat([ts[f"{int(yr):04d}-{m:02d}"] for m in months])

            # 每个月至少 28 天
            if len(combined) < len(months) * 28:
                continue

            val = combined.mean() if method == "mean" else combined.min()
            results[yr] = val

        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def plot_custom_relationship(exp_name, ep_nc, o3_nc, txt_path,
                             fz_season="DJF", fz_method="mean",
                             o3_season="MA", o3_method="mean",
                             lat_band=(40, 80), apply_o3_5d=True):

    print(f"[{exp_name}] Analyzing Fz({fz_season}, {fz_method}) vs O3({o3_season}, {o3_method})...")

    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        print("  Missing file(s). Skip.")
        return

    # 1. 加载并预处理数据
    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"

    # 100 hPa daily
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    # 若存在 lat 维，则统一做 40–80N 加权平均
    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 2. 提取指定季节和方法的统计量
    fz_stat = get_seasonal_data(fz_raw, fz_season, method=fz_method)

    # ✅ 关键修正：
    # 若 Fz 使用的是跨年季节（如 DJF / NDJ），则删除连接年 104
    # 因为该年的季节值是由 run001 的年末 + run002 的年初拼接出来的“人工季节”
    if is_cross_year_season(fz_season):
        if BRIDGE_YEAR in fz_stat.year.values:
            fz_stat = fz_stat.sel(year=fz_stat.year != BRIDGE_YEAR)
            print(f"  Removed bridge year {BRIDGE_YEAR} from cross-year Fz season ({fz_season}).")

    # O3: 5-day running mean before seasonal stats
    if apply_o3_5d:
        da_o3 = da_o3.rolling(time=5, center=True, min_periods=5).mean()

    o3_stat = get_seasonal_data(da_o3, o3_season, method=o3_method)

    # 3. 对齐年份
    common_years = np.intersect1d(fz_stat.year.values, o3_stat.year.values)
    x_raw = fz_stat.sel(year=common_years)
    y_raw = o3_stat.sel(year=common_years)

    if len(common_years) < 3:
        print("  Too few common years after filtering. Skip.")
        ds_ep.close()
        return

    # 4. 标准化
    x = (x_raw - x_raw.mean()) / x_raw.std()
    y = (y_raw - y_raw.mean()) / y_raw.std()

    # 5. 绘图
    plt.figure(figsize=(7, 6), dpi=120)
    plt.grid(True, linestyle=":", alpha=0.6)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.axvline(0, color="k", lw=0.8, alpha=0.5)

    key_colors = ['#d62728', '#1f77b4', '#2ca02c', '#ff7f0e', '#9467bd']
    key_years = np.loadtxt(txt_path, dtype=int)[:3] if os.path.exists(txt_path) else []

    # 背景点
    bg_mask = ~np.isin(x.year.values, key_years)
    plt.scatter(
        x.values[bg_mask], y.values[bg_mask],
        color="black", alpha=0.8, s=40, edgecolors="none",
        label="Other Years"
    )

    # 高亮关键年
    for i, yr in enumerate(np.atleast_1d(key_years)):
        if yr in x.year.values:
            idx = np.where(x.year.values == yr)[0][0]
            plt.scatter(
                x.values[idx], y.values[idx],
                color=key_colors[i % len(key_colors)],
                s=40, edgecolors="k", linewidths=0.8,
                label=f"Year {int(yr):04d}", zorder=10
            )

    plt.legend(loc="center left", bbox_to_anchor=(1, 0.5), frameon=False, fontsize=9)

    r, p = pearsonr(x.values, y.values)
    plt.text(
        0.05, 0.95,
        f"r = {r:.3f}\np = {p:.2e}\nN = {len(x)}",
        transform=plt.gca().transAxes,
        va="top",
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.7, edgecolor="none")
    )

    plt.xlim(-5, 5)
    plt.ylim(-5, 5)
    plt.xlabel(f"{fz_season} $F_z$ at 100hPa ({fz_method}, $\\sigma$)", fontsize=10)
    plt.ylabel(f"{o3_season} O3 Column ({o3_method}, $\\sigma$)", fontsize=10)
    plt.title(f"{exp_name} | {fz_season} Fz vs {o3_season} O3", loc="left", fontweight="bold")

    out_dir = os.path.join(PLOT_BASE, exp_name, "EP_O3_Custom")
    os.makedirs(out_dir, exist_ok=True)
    save_path = os.path.join(out_dir, f"Scatter_{fz_season}_{fz_method}_vs_{o3_season}_{o3_method}.png")

    plt.savefig(save_path, bbox_inches="tight", dpi=300)
    plt.show()
    print("Saved:", save_path)

    ds_ep.close()

# ---------------- 执行示例 ----------------
configs = [
    {
        "fz_season": "DJF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    },
    {
        "fz_season": "JF",
        "fz_method": "mean",
        "o3_season": "MA",
        "o3_method": "min"
    }
]

tasks = [
    (
        "B2000WCN_NOCOUPL",
        FILE_NOCOUPL_EP,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/lowest10_years_sorted_30_70hPa.txt")
    )
]

for name, ep, o3, txt in tasks:
    for cfg in configs:
        plot_custom_relationship(name, ep, o3, txt, **cfg)

In [ ]:
# ============================================================
# Block D (NOCOUPL): Sliding-window correlation curves (ALL years)
# 目标：
# 1) B2000WCN_NOCOUPL 单独分析
# 2) 正确跳过连接年 104（仅对 12 月起始窗口）
# 3) 90-day 和 60-day 分开作图
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_NOCOUPL_EP = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    """
    窗口起点：12-01 到 02-01，逐日滑动
    """
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    返回按 year 的 MA minimum O3（目标变量）
    """
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口 Fz 均值
    - 若起点在 12 月：窗口从 (year-1)-12-dd 开始
    - 若起点在 1/2 月：窗口从 year-mm-dd 开始
    """
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def compute_r_curve_nocoupl_all(window_days, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    if not os.path.exists(FILE_NOCOUPL_EP):
        print("[B2000WCN_NOCOUPL] Missing EP file.")
        return None, None

    o3_nc = os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc")
    if not os.path.exists(o3_nc):
        print("[B2000WCN_NOCOUPL] Missing O3 file.")
        return None, None

    ds_ep = xr.open_dataset(FILE_NOCOUPL_EP)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)

        # ✅ NOCOUPL 的连接年修正：
        # 仅对 12 月起始窗口删除 year=104
        if start_month == 12:
            if BRIDGE_YEAR in x_raw.year.values:
                x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

        common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
        x_use = x_raw.sel(year=common_years)
        y_use = y_raw.sel(year=common_years)

        if len(common_years) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(x_use.values, y_use.values)
        except:
            r = np.nan

        r_list.append(r)

    ds_ep.close()
    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="ALL"):
    out_dir = os.path.join(PLOT_BASE, "B2000WCN_NOCOUPL", "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"NOCOUPL Ozone Long Run | {window_days}-day sliding-window correlation with MA minimum O3 ({suffix})",
        loc="left", fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_r_curve_nocoupl_all(window_days=wd)
    if labels is not None:
        plot_single_curve(labels, r_list, window_days=wd, suffix="ALL")

In [ ]:
# ============================================================
# Block E (NOCOUPL): Sliding-window correlation curves (LOW 25% years)
# 目标：
# 1) B2000WCN_NOCOUPL 单独分析
# 2) 仅使用 low25pct_years_30_70hPa
# 3) 正确跳过连接年 104（仅对 12 月起始窗口）
# 4) 90-day 和 60-day 分开作图
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_NOCOUPL_EP = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"
DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

def build_window_starts():
    starts = []
    for d in range(1, 32):
        starts.append((12, d))
    for d in range(1, 32):
        starts.append((1, d))
    starts.append((2, 1))
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def compute_r_curve_nocoupl_low25(window_days, lat_band=(40, 80), apply_o3_5d=True, min_n=5):
    if not os.path.exists(FILE_NOCOUPL_EP):
        print("[B2000WCN_NOCOUPL] Missing EP file.")
        return None, None

    o3_nc = os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc")
    low25_txt = os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/low25pct_years_30_70hPa.txt")

    if not (os.path.exists(o3_nc) and os.path.exists(low25_txt)):
        print("[B2000WCN_NOCOUPL] Missing O3 or low25 file.")
        return None, None

    ds_ep = xr.open_dataset(FILE_NOCOUPL_EP)
    da_o3 = xr.open_dataarray(o3_nc)

    low25_years = np.loadtxt(low25_txt, dtype=int)
    low25_years = np.atleast_1d(low25_years)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)

    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    for start_month, start_day in starts:
        x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)

        # ✅ NOCOUPL 的连接年修正：
        # 仅对 12 月起始窗口删除 year=104
        if start_month == 12:
            if BRIDGE_YEAR in x_raw.year.values:
                x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

        common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)
        use_years = np.intersect1d(common_years, low25_years)

        x_use = x_raw.sel(year=use_years)
        y_use = y_raw.sel(year=use_years)

        if len(use_years) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(x_use.values, y_use.values)
        except:
            r = np.nan

        r_list.append(r)

    ds_ep.close()
    return labels, r_list

def plot_single_curve(labels, r_list, window_days, suffix="LOW25"):
    out_dir = os.path.join(PLOT_BASE, "B2000WCN_NOCOUPL", "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5), dpi=140)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))
    plt.plot(x, r_list, lw=2.2, marker="o", ms=3)

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"NOCOUPL Ozone Long Run | {window_days}-day sliding-window correlation with MA minimum O3 ({suffix})",
        loc="left", fontweight="bold"
    )

    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_{window_days}d_vs_MAmin_{suffix}.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
for wd in WINDOW_DAYS_LIST:
    labels, r_list = compute_r_curve_nocoupl_low25(window_days=wd)
    if labels is not None:
        plot_single_curve(labels, r_list, window_days=wd, suffix="LOW25")

In [ ]:
# ============================================================
# Block F: Combined comparison plots
# 目标：
# 生成两张图（90-day / 60-day），每张图包含 4 条线：
#   1) Interactive ALL
#   2) Interactive LOW25
#   3) NOCOUPL ALL
#   4) NOCOUPL LOW25
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]

# ---------------- 通用工具函数 ----------------
def build_window_starts():
    starts = []
    for d in range(1, 32):   # Dec 1-31
        starts.append((12, d))
    for d in range(1, 32):   # Jan 1-31
        starts.append((1, d))
    starts.append((2, 1))    # Feb 1
    return starts

def get_ma_min_o3_series(o3_da, apply_o3_5d=True):
    """
    目标变量：MA 的 5-day running mean minimum O3
    """
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    ts = o3_da.to_series()
    all_years = np.unique(o3_da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            seg = pd.concat([
                ts[f"{int(yr):04d}-03"],
                ts[f"{int(yr):04d}-04"]
            ])
            if len(seg) < 58:
                continue
            results[yr] = seg.min()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def get_window_mean_series(da, start_month, start_day, window_days):
    """
    对每个 target year，取固定长度滑动窗口 Fz 均值
    - 若起点在 12 月：窗口从 (year-1)-12-dd 开始
    - 若起点在 1/2 月：窗口从 year-mm-dd 开始
    """
    ts = da.to_series()
    all_years = np.unique(da.time.dt.year.values)

    results = {}
    for yr in all_years:
        try:
            start_year = int(yr) - 1 if start_month == 12 else int(yr)
            start_key = f"{start_year:04d}-{start_month:02d}-{start_day:02d}"

            seg = ts[start_key:]
            if len(seg) < window_days:
                continue

            win = seg.iloc[:window_days]
            if len(win) < window_days:
                continue

            results[yr] = win.mean()
        except:
            continue

    return pd.Series(results).to_xarray().rename({"index": "year"})

def load_fz_and_o3(ep_nc, o3_nc, lat_band=(40, 80), apply_o3_5d=True):
    """
    读取并返回：
    - fz_raw: 100 hPa, 40-80N mean
    - y_raw: MA minimum O3
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        return None, None, None

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    y_raw = get_ma_min_o3_series(da_o3, apply_o3_5d=apply_o3_5d)
    return ds_ep, fz_raw, y_raw

def make_df_for_window(exp_name, fz_raw, y_raw, start_month, start_day, window_days,
                       low25_years=None, is_bridge_case=False):
    """
    生成给定窗口下的样本表：
    columns = [exp, year, x_raw, y_raw]
    """
    x_raw = get_window_mean_series(fz_raw, start_month, start_day, window_days)

    # 仅对拼接型实验，且仅 12 月起点，删除连接年 104
    if is_bridge_case and (start_month == 12):
        if BRIDGE_YEAR in x_raw.year.values:
            x_raw = x_raw.sel(year=x_raw.year != BRIDGE_YEAR)

    common_years = np.intersect1d(x_raw.year.values, y_raw.year.values)

    if low25_years is not None:
        common_years = np.intersect1d(common_years, low25_years)

    if len(common_years) == 0:
        return pd.DataFrame(columns=["exp", "year", "x_raw", "y_raw"])

    x_use = x_raw.sel(year=common_years)
    y_use = y_raw.sel(year=common_years)

    return pd.DataFrame({
        "exp": exp_name,
        "year": x_use.year.values.astype(int),
        "x_raw": x_use.values.astype(float),
        "y_raw": y_use.values.astype(float),
    })

def compute_curve_from_df_builder(window_days, mode="interactive_all", min_n=5):
    """
    返回一个 r 曲线
    mode:
      - interactive_all
      - interactive_low25
      - nocoupl_all
      - nocoupl_low25
    """
    starts = build_window_starts()
    labels = [f"{m:02d}-{d:02d}" for m, d in starts]
    r_list = []

    # --- 预加载数据 ---
    # Interactive: BWCN + B2000WCN
    ds_bwcn, fz_bwcn, y_bwcn = load_fz_and_o3(
        FILE_BWCN_EP,
        os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc")
    )
    ds_b2000, fz_b2000, y_b2000 = load_fz_and_o3(
        FILE_B2000_EP,
        os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc")
    )

    # NOCOUPL
    ds_noc, fz_noc, y_noc = load_fz_and_o3(
        FILE_NOCOUPL_EP,
        os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc")
    )

    if mode == "interactive_low25":
        low25_bwcn = np.atleast_1d(np.loadtxt(os.path.join(DATA_BASE, "BWCN/low25pct_years_30_70hPa.txt"), dtype=int))
        low25_b2000 = np.atleast_1d(np.loadtxt(os.path.join(DATA_BASE, "B2000WCN/low25pct_years_30_70hPa.txt"), dtype=int))
    else:
        low25_bwcn = low25_b2000 = None

    if mode == "nocoupl_low25":
        low25_noc = np.atleast_1d(np.loadtxt(os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/low25pct_years_30_70hPa.txt"), dtype=int))
    else:
        low25_noc = None

    for start_month, start_day in starts:
        if mode in {"interactive_all", "interactive_low25"}:
            df_bwcn = make_df_for_window(
                "BWCN", fz_bwcn, y_bwcn,
                start_month, start_day, window_days,
                low25_years=low25_bwcn,
                is_bridge_case=False
            )
            df_b2000 = make_df_for_window(
                "B2000WCN", fz_b2000, y_b2000,
                start_month, start_day, window_days,
                low25_years=low25_b2000,
                is_bridge_case=True
            )
            df_all = pd.concat([df_bwcn, df_b2000], ignore_index=True).dropna()

        elif mode in {"nocoupl_all", "nocoupl_low25"}:
            df_all = make_df_for_window(
                "B2000WCN_NOCOUPL", fz_noc, y_noc,
                start_month, start_day, window_days,
                low25_years=low25_noc,
                is_bridge_case=True
            ).dropna()

        else:
            raise ValueError(f"Unknown mode: {mode}")

        if len(df_all) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(df_all["x_raw"].values, df_all["y_raw"].values)
        except:
            r = np.nan

        r_list.append(r)

    # 关闭数据集
    for ds in [ds_bwcn, ds_b2000, ds_noc]:
        if ds is not None:
            ds.close()

    return labels, r_list

def plot_combined_comparison(labels, curves_dict, window_days):
    """
    同一张图放 4 条线：
      - Interactive ALL
      - Interactive LOW25
      - NOCOUPL ALL
      - NOCOUPL LOW25
    """
    out_dir = os.path.join(PLOT_BASE, "COMBINED_COMPARE", "EP_O3_SlidingR")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5.5), dpi=150)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    x = np.arange(len(labels))

    plt.plot(x, curves_dict["interactive_all"],   lw=2.2, marker="o", ms=3, label="Interactive ALL")
    plt.plot(x, curves_dict["interactive_low25"], lw=2.2, marker="o", ms=3, label="Interactive LOW25")
    plt.plot(x, curves_dict["nocoupl_all"],       lw=2.2, marker="o", ms=3, label="NOCOUPL ALL")
    plt.plot(x, curves_dict["nocoupl_low25"],     lw=2.2, marker="o", ms=3, label="NOCOUPL LOW25")

    tick_idx = np.arange(0, len(labels), 5)
    plt.xticks(tick_idx, [labels[i] for i in tick_idx], rotation=45)

    plt.ylim(-1, 1)
    plt.xlim(0, len(labels) - 1)
    plt.ylabel("Pearson r")
    plt.xlabel("Window start date (MM-DD)")
    plt.title(
        f"Interactive vs NOCOUPL | {window_days}-day sliding-window correlation with MA minimum O3",
        loc="left", fontweight="bold"
    )

    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()

    save_path = os.path.join(out_dir, f"SlidingR_COMPARE_{window_days}d_ALL_LOW25.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 执行 ----------------
for wd in WINDOW_DAYS_LIST:
    labels, curve_inter_all = compute_curve_from_df_builder(wd, mode="interactive_all")
    _,      curve_inter_l25 = compute_curve_from_df_builder(wd, mode="interactive_low25")
    _,      curve_noc_all   = compute_curve_from_df_builder(wd, mode="nocoupl_all")
    _,      curve_noc_l25   = compute_curve_from_df_builder(wd, mode="nocoupl_low25")

    curves = {
        "interactive_all": curve_inter_all,
        "interactive_low25": curve_inter_l25,
        "nocoupl_all": curve_noc_all,
        "nocoupl_low25": curve_noc_l25,
    }

    plot_combined_comparison(labels, curves, window_days=wd)

In [ ]:
# ============================================================
# Block G: Event-anchored sliding-window correlation (anchor = each year's MA O3 minimum day)
# 目标：
# - 对每年，用 MA 中 5-day running mean O3 最低值那一天作为锚点
# - 从锚点往前 lead=n 天（n=150..60），取从该日起始的 90d / 60d Fz window
# - 计算 Fz window mean 与 MA minimum O3 的跨年相关
# - 输出 2 张图（90d / 60d），每张图 4 条线：
#     1) Interactive ALL
#     2) Interactive LOW25
#     3) NOCOUPL ALL
#     4) NOCOUPL LOW25
# 注意：
# - 正确处理 B2000WCN 与 B2000WCN_NOCOUPL 的连接年问题
# - BWCN 不做连接年修正
# ============================================================
import os
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr

# ---------------- 基础配置 ----------------
FILE_BWCN_EP     = "/home/weiji/restart_exam/code/20260122StructureAnalysis/result/EPFLUX_BWCN002_stdplev_daily_lat_ALL.nc"
FILE_B2000_EP    = "/mnt/soclim0/public_data/weiji/B2000WCN001002/EPflux_daily/EPFLUX_206yr_Daily_Series_Full.nc"
FILE_NOCOUPL_EP  = "/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002/EPflux_daily/EPFLUX_205yr_Daily_Series_Full.nc"

DATA_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
PLOT_BASE = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/plots"

BRIDGE_YEAR = 104
WINDOW_DAYS_LIST = [90, 60]
LEAD_DAYS = list(range(150, 59, -1))   # 150,149,...,60

# ---------------- 工具函数 ----------------
def get_ma_min_o3_and_anchor(o3_da, apply_o3_5d=True):
    """
    对每个年份，返回：
    - y_min: MA 中 5-day running mean O3 的最低值
    - anchor_idx: 该最低值对应的全局时间索引位置（在对齐后的时间轴上）
    """
    if apply_o3_5d:
        o3_da = o3_da.rolling(time=5, center=True, min_periods=5).mean()

    years = o3_da.time.dt.year.values
    months = o3_da.time.dt.month.values

    year_info = {}
    unique_years = np.unique(years)

    for yr in unique_years:
        mask = (years == yr) & np.isin(months, [3, 4])
        idxs = np.where(mask)[0]
        if len(idxs) < 58:
            continue

        vals = o3_da.isel(time=idxs).values
        finite = np.isfinite(vals)
        if finite.sum() < 58:
            continue

        # 用 finite 子集做 argmin，再映射回全局索引
        valid_local = np.where(finite)[0]
        valid_vals = vals[valid_local]
        best_local_in_valid = int(np.argmin(valid_vals))
        best_local = valid_local[best_local_in_valid]

        anchor_idx = int(idxs[best_local])
        y_min = float(vals[best_local])

        year_info[int(yr)] = {
            "anchor_idx": anchor_idx,
            "y_min": y_min,
        }

    return o3_da, year_info

def prepare_experiment(exp_name, ep_nc, o3_nc, low25_txt=None, lat_band=(40, 80), apply_o3_5d=True):
    """
    读取并预处理一个实验，返回：
    - fz_raw: 100 hPa, 40–80N mean（1D time）
    - year_info: 每年 anchor_idx 与 y_min
    - low25_years: set
    - bridge_boundary_idx: 若为拼接实验，则为第一个 year=104 的时间索引；否则 None
    """
    if not (os.path.exists(ep_nc) and os.path.exists(o3_nc)):
        raise FileNotFoundError(f"[{exp_name}] Missing EP/O3 file.")

    ds_ep = xr.open_dataset(ep_nc)
    da_o3 = xr.open_dataarray(o3_nc)

    var_fz = "Fz" if "Fz" in ds_ep else "EP2_cart"
    fz_raw = -1.0 * ds_ep[var_fz].sel(plev=10000, method="nearest")

    if "lat" in fz_raw.coords:
        fz_raw = fz_raw.sel(lat=slice(lat_band[0], lat_band[1]))
        weights = np.cos(np.deg2rad(fz_raw.lat))
        fz_raw = fz_raw.weighted(weights).mean("lat")

    # 时间对齐，确保 anchor_idx 能直接用于 fz_raw
    fz_raw, o3_aligned = xr.align(fz_raw, da_o3, join="inner")

    # 显式 load，后面用 numpy 索引更稳
    fz_raw = fz_raw.load()
    o3_aligned = o3_aligned.load()

    o3_5d, year_info = get_ma_min_o3_and_anchor(o3_aligned, apply_o3_5d=apply_o3_5d)

    if low25_txt is not None and os.path.exists(low25_txt):
        low25_years = set(np.atleast_1d(np.loadtxt(low25_txt, dtype=int)).tolist())
    else:
        low25_years = set()

    # 仅拼接型实验需要边界位置
    bridge_boundary_idx = None
    if exp_name in {"B2000WCN", "B2000WCN_NOCOUPL"}:
        years = fz_raw.time.dt.year.values
        idxs_104 = np.where(years == BRIDGE_YEAR)[0]
        if len(idxs_104) > 0:
            bridge_boundary_idx = int(idxs_104[0])  # 第一个 0104-01-01

    ds_ep.close()

    return {
        "exp": exp_name,
        "fz_raw": fz_raw,
        "year_info": year_info,
        "low25_years": low25_years,
        "bridge_boundary_idx": bridge_boundary_idx,
    }

def build_samples_event_anchored(prep, lead_days, window_days, use_low25=False):
    """
    基于 anchor-day 构建样本：
    - 窗口起点 = anchor_idx - lead_days
    - 窗口 = [start_idx, start_idx + window_days)
    """
    fz = prep["fz_raw"].values
    ntime = len(fz)
    boundary = prep["bridge_boundary_idx"]

    rows = []

    for yr, info in prep["year_info"].items():
        if use_low25 and (yr not in prep["low25_years"]):
            continue

        anchor_idx = info["anchor_idx"]
        y_min = info["y_min"]

        start_idx = anchor_idx - lead_days
        end_idx = start_idx + window_days  # python slice end-exclusive

        if start_idx < 0 or end_idx > ntime:
            continue

        # ✅ 精确的连接年修正：
        # 仅对拼接实验，且仅当该窗口真的跨过边界时，剔除这个样本
        if boundary is not None:
            if (start_idx < boundary) and (end_idx > boundary):
                # 该窗口横跨 run001 -> run002 边界，跳过
                continue

        x_vals = fz[start_idx:end_idx]
        if np.isfinite(x_vals).sum() < window_days:
            continue

        x_mean = float(np.nanmean(x_vals))

        rows.append({
            "exp": prep["exp"],
            "year": int(yr),
            "x_raw": x_mean,
            "y_raw": float(y_min),
        })

    return pd.DataFrame(rows)

def compute_event_anchored_curve(prep_inter_bwcn, prep_inter_b2000, prep_noc, window_days, mode, min_n=5):
    """
    mode:
      - interactive_all
      - interactive_low25
      - nocoupl_all
      - nocoupl_low25
    返回：
      - leads
      - r_list
    """
    r_list = []

    for lead in LEAD_DAYS:
        if mode == "interactive_all":
            df1 = build_samples_event_anchored(prep_inter_bwcn,  lead, window_days, use_low25=False)
            df2 = build_samples_event_anchored(prep_inter_b2000, lead, window_days, use_low25=False)
            df_all = pd.concat([df1, df2], ignore_index=True).dropna()

        elif mode == "interactive_low25":
            df1 = build_samples_event_anchored(prep_inter_bwcn,  lead, window_days, use_low25=True)
            df2 = build_samples_event_anchored(prep_inter_b2000, lead, window_days, use_low25=True)
            df_all = pd.concat([df1, df2], ignore_index=True).dropna()

        elif mode == "nocoupl_all":
            df_all = build_samples_event_anchored(prep_noc, lead, window_days, use_low25=False).dropna()

        elif mode == "nocoupl_low25":
            df_all = build_samples_event_anchored(prep_noc, lead, window_days, use_low25=True).dropna()

        else:
            raise ValueError(f"Unknown mode: {mode}")

        if len(df_all) < min_n:
            r_list.append(np.nan)
            continue

        try:
            r, _ = pearsonr(df_all["x_raw"].values, df_all["y_raw"].values)
        except:
            r = np.nan

        r_list.append(r)

    return LEAD_DAYS, r_list

def plot_event_anchored_compare(leads, curves_dict, window_days):
    """
    画一张对比图：
      - Interactive ALL
      - Interactive LOW25
      - NOCOUPL ALL
      - NOCOUPL LOW25
    x 轴 = lead days before anchor
    """
    out_dir = os.path.join(PLOT_BASE, "COMBINED_COMPARE", "EP_O3_EventAnchored")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure(figsize=(12, 5.5), dpi=150)
    plt.axhline(0, color="k", lw=0.8, alpha=0.5)
    plt.grid(True, linestyle=":", alpha=0.4)

    plt.plot(leads, curves_dict["interactive_all"],   lw=2.2, marker="o", ms=3, label="Interactive ALL")
    plt.plot(leads, curves_dict["interactive_low25"], lw=2.2, marker="o", ms=3, label="Interactive LOW25")
    plt.plot(leads, curves_dict["nocoupl_all"],       lw=2.2, marker="o", ms=3, label="NOCOUPL ALL")
    plt.plot(leads, curves_dict["nocoupl_low25"],     lw=2.2, marker="o", ms=3, label="NOCOUPL LOW25")

    plt.ylim(-1, 1)
    plt.xlim(max(leads), min(leads))  # 从 150 到 60，按你的要求从大到小显示
    plt.ylabel("Pearson r")
    plt.xlabel("Lead days before each year's MA O3-min anchor day")
    plt.title(
        f"Event-anchored comparison | {window_days}-day Fz window vs MA minimum O3",
        loc="left", fontweight="bold"
    )

    plt.legend(frameon=False, ncol=2)
    plt.tight_layout()

    save_path = os.path.join(out_dir, f"EventAnchored_COMPARE_{window_days}d_ALL_LOW25.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    print("Saved:", save_path)

# ---------------- 预处理实验 ----------------
prep_bwcn = prepare_experiment(
    "BWCN",
    FILE_BWCN_EP,
    os.path.join(DATA_BASE, "BWCN/O3_pc_BWCN_30_70hPa.nc"),
    low25_txt=os.path.join(DATA_BASE, "BWCN/low25pct_years_30_70hPa.txt")
)

prep_b2000 = prepare_experiment(
    "B2000WCN",
    FILE_B2000_EP,
    os.path.join(DATA_BASE, "B2000WCN/O3_pc_B2000WCN_30_70hPa.nc"),
    low25_txt=os.path.join(DATA_BASE, "B2000WCN/low25pct_years_30_70hPa.txt")
)

prep_noc = prepare_experiment(
    "B2000WCN_NOCOUPL",
    FILE_NOCOUPL_EP,
    os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/O3_pc_B2000WCN_NOCOUPL_30_70hPa.nc"),
    low25_txt=os.path.join(DATA_BASE, "B2000WCN_NOCOUPL/low25pct_years_30_70hPa.txt")
)

# ---------------- 执行：90d 和 60d 各一张 ----------------
for wd in WINDOW_DAYS_LIST:
    leads, curve_inter_all = compute_event_anchored_curve(prep_bwcn, prep_b2000, prep_noc, wd, mode="interactive_all")
    _,     curve_inter_l25 = compute_event_anchored_curve(prep_bwcn, prep_b2000, prep_noc, wd, mode="interactive_low25")
    _,     curve_noc_all   = compute_event_anchored_curve(prep_bwcn, prep_b2000, prep_noc, wd, mode="nocoupl_all")
    _,     curve_noc_l25   = compute_event_anchored_curve(prep_bwcn, prep_b2000, prep_noc, wd, mode="nocoupl_low25")

    curves = {
        "interactive_all": curve_inter_all,
        "interactive_low25": curve_inter_l25,
        "nocoupl_all": curve_noc_all,
        "nocoupl_low25": curve_noc_l25,
    }

    plot_event_anchored_compare(leads, curves, window_days=wd)